## Computer Vision - Part 5

### 6D Object Pose Estimation for Robotics

### Lab 1: Build a Pose Matrix

Construct a homogeneous 4×4 transformation matrix from scratch using NumPy:

In [1]:
import numpy as np

T = np.eye(4) # Start with identity matrix
T[0, 3] = 0.25  # x translation: 25 cm
T[1, 3] = 0.10  # y translation: 10 cm
T[2, 3] = 0.40  # z translation: 40 cm
# T[:3, :3] holds the rotation matrix (identity = no rotation)
print("Pose Matrix:\n", T)
print("Rotation Matrix:\n", T[:3, :3])

Pose Matrix:
 [[1.   0.   0.   0.25]
 [0.   1.   0.   0.1 ]
 [0.   0.   1.   0.4 ]
 [0.   0.   0.   1.  ]]
Rotation Matrix:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


### Lab 2: Read Pose from an AI Model

A typical AI model returns pose as a Python dictionary:

In [2]:
pose = {"x": 0.32, # meters 
        "y": -0.05, 
        "z": 0.41, 
        "roll": 0.10, # radians
        "pitch": -0.02, 
        "yaw": 1.57 # ~90 degrees
        }
print(f"Position: ({pose['x']}, {pose['y']}, {pose['z']})")
print(f"Orientation (RPY): ({pose['roll']}, {pose['pitch']}, {pose['yaw']})")

Position: (0.32, -0.05, 0.41)
Orientation (RPY): (0.1, -0.02, 1.57)


### Lab 3: Compute Box Center from Detection

Bridge detection output to pose estimation by computing the 2D region of interest:

In [3]:
def pose_model_predict(roi):
    # Dummy function to simulate pose estimation
    return {"x": 0.30, "y": 0.00, "z": 0.40, "roll": 0.05, "pitch": 0.00, "yaw": 1.50}

In [4]:
# Assume x1, y1 = top-left corner; x2, y2 = bottom-right corner
x1, y1 = 100, 150
x2, y2 = 200, 250
image = np.zeros((300, 300, 3), dtype=np.uint8) # Dummy image

u = int((x1 + x2) / 2)
v = int((y1 + y2) / 2)
# Crop the ROI around the detected object
roi = image[y1:y2, x1:x2]
# Feed ROI into pose estimation model
pose = pose_model_predict(roi)

print(f"Bounding Box Center: (u: {u}, v: {v})")
print(f"ROI Shape: {roi.shape}")
print(f"Estimated Pose: {pose}")

Bounding Box Center: (u: 150, v: 200)
ROI Shape: (100, 100, 3)
Estimated Pose: {'x': 0.3, 'y': 0.0, 'z': 0.4, 'roll': 0.05, 'pitch': 0.0, 'yaw': 1.5}


### Lab 4: solvePnP with OpenCV

Apply the classical PnP solver to known 2D–3D correspondences:

In [5]:
import cv2

# Nx3 array of 3D model points 
object_points = np.array([
    [0.0, 0.0, 0.0],   # Point 1
    [0.1, 0.0, 0.0],   # Point 2
    [0.1, 0.1, 0.0],   # Point 3
    [0.0, 0.1, 0.0],    # Point 4 
    ], dtype=np.float32)

# Nx2 array of 2D image points
image_points = np.array([
    [150, 200],  # Point 1
    [250, 200],  # Point 2
    [250, 300],  # Point 3
    [150, 300]   # Point 4
], dtype=np.float32)

# 3x3 intrinsic matrix K
camera_matrix = np.array([
    [800, 0, 320],  # fx, 0, cx
    [0, 800, 240],  # 0, fy, cy
    [0, 0, 1]       # 0, 0, 1
], dtype=np.float32)

# lens distortion coefficients (distortion is zero for this example)
dist_coeffs = np.zeros(4, dtype=np.float32)

ret, rvec, tvec = cv2.solvePnP(object_points, image_points, camera_matrix, dist_coeffs)
print("PnP Solution Found:", ret)
print("Rotation Vector:\n", rvec)
print("Translation Vector:\n", tvec)

# Convert rotation vector to rotation matrix
R, _ = cv2.Rodrigues(rvec)
print("Rotation Matrix:\n", R)

PnP Solution Found: True
Rotation Vector:
 [[1.95642242e-09]
 [7.92560232e-10]
 [1.72182569e-10]]
Translation Vector:
 [[-0.17      ]
 [-0.04      ]
 [ 0.80000001]]
Rotation Matrix:
 [[ 1.00000000e+00 -1.72182569e-10  7.92560232e-10]
 [ 1.72182569e-10  1.00000000e+00 -1.95642242e-09]
 [-7.92560232e-10  1.95642242e-09  1.00000000e+00]]


### Lab 5: Full Integration Pipeline Idea

In [6]:
''' def full_pipeline(image):
    detector = ObjectDetector()  # Placeholder for trained object detector
    pose_estimator = PoseEstimator()  # Placeholder for trained pose estimator
    T_robot_camera = np.eye(4)  # Placeholder for hand-eye calibration matrix
    robot_arm = RobotArm()  # Placeholder for robot arm interface   

# Step 1: Detect the object using a trained detector
bbox, class_id = detector.predict(image)
# Step 2: Estimate 6D pose relative to the camera frame
pose_in_camera = pose_estimator.predict(image, bbox)
# Step 3: Convert pose from camera frame to robot base frame
# Requires hand-eye calibration matrix T_robot_camera
pose_in_robot = T_robot_camera @ pose_in_camera
# Step 4: Send target pose to IK solver / motion planner
robot_arm.move_to(pose_in_robot)

'''

' def full_pipeline(image):\n    detector = ObjectDetector()  # Placeholder for trained object detector\n    pose_estimator = PoseEstimator()  # Placeholder for trained pose estimator\n    T_robot_camera = np.eye(4)  # Placeholder for hand-eye calibration matrix\n    robot_arm = RobotArm()  # Placeholder for robot arm interface   \n\n# Step 1: Detect the object using a trained detector\nbbox, class_id = detector.predict(image)\n# Step 2: Estimate 6D pose relative to the camera frame\npose_in_camera = pose_estimator.predict(image, bbox)\n# Step 3: Convert pose from camera frame to robot base frame\n# Requires hand-eye calibration matrix T_robot_camera\npose_in_robot = T_robot_camera @ pose_in_camera\n# Step 4: Send target pose to IK solver / motion planner\nrobot_arm.move_to(pose_in_robot)\n\n'

### Assignment: Implement the full pipeline in code:

- Use YOLO to detect object from video (or webcam)
- For each bounding box, use a small CNN to predict 8 keypoints on the object
- Use solvePnP to compute 6D pose from the keypoints
- Draw 3D bounding box on the image to visualize the pose estimation results

In [7]:
# Step 1: Build small CNN to predict 8 keypoints on the object (dummy example)
import tensorflow as tf
from tensorflow.keras import layers, models

def build_model():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),

        # 8 keypoints → (x1,y1,...x8,y8)
        layers.Dense(16, activation='linear')
    ])

    model.compile(
        optimizer='adam',
        loss='mse'
    )
    return model

In [8]:
# Step 2: 3D object keypoints important for pose estimation
object_points_3d = np.array([
    [-1, -0.5,  2],  # front-left-bottom
    [ 1, -0.5,  2],  # front-right-bottom
    [ 1,  0.5,  2],  # front-right-top
    [-1,  0.5,  2],  # front-left-top
    [-1, -0.5,  0],  # back-left-bottom
    [ 1, -0.5,  0],  # back-right-bottom
    [ 1,  0.5,  0],  # back-right-top
    [-1,  0.5,  0],  # back-left-top
], dtype=np.float32)

In [9]:
# Step 3: Camera matrix (required for solvePnP)
focal_length = 800
cx, cy = 640/2, 480/2

camera_matrix = np.array([[focal_length, 0, cx], 
                          [0, focal_length, cy], 
                          [0, 0, 1]], dtype=np.float32)

dist_coeffs = np.zeros((4,1))

In [ ]:
# The main loop would look like this
import cv2
from ultralytics import YOLO
import numpy as np
import tensorflow as tf

# Load models
yolo = YOLO("yolov8n.pt")
cnn = build_model()

cap = cv2.VideoCapture("car.mp4")


dist_coeffs = np.zeros((4,1))

# Video loop
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo(frame)

    for r in results:
        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            # resize for CNN
            crop_resized = cv2.resize(crop, (128, 128))
            crop_input = crop_resized / 255.0
            crop_input = np.expand_dims(crop_input, axis=0)

            # predict keypoints
            preds = cnn.predict(crop_input)[0]

            keypoints_2d = []

            for i in range(0, 16, 2):
                x = preds[i] * (x2 - x1) + x1
                y = preds[i+1] * (y2 - y1) + y1
                keypoints_2d.append([x, y])

            keypoints_2d = np.array(keypoints_2d, dtype=np.float32)

            # solvePnP
            success, rvec, tvec = cv2.solvePnP(
                object_points_3d,
                keypoints_2d,
                camera_matrix,
                dist_coeffs)

            if success:
                # project 3D box
                axis = np.float32([
                    [2,0,0], [0,2,0], [0,0,2]
                ])

                imgpts, _ = cv2.projectPoints(
                    axis, rvec, tvec, camera_matrix, dist_coeffs
                )

                origin = tuple(keypoints_2d[0].astype(int))

                for p in imgpts:
                    p = tuple(p.ravel().astype(int))
                    cv2.line(frame, origin, p, (0,255,0), 2)

            # draw bbox
            cv2.rectangle(frame, (x1,y1), (x2,y2), (255,0,0), 2)

    cv2.imshow("Pose Estimation", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 9 cars, 1 stop sign, 72.2ms
Speed: 3.6ms preprocess, 72.2ms inference, 14.3ms postprocess per image at shape (1, 3, 384, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step

0: 384x640 9 cars, 1 stop sign, 40.4ms
Speed: 0.9ms preprocess, 40.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/

: 